# Cleaning My Scraped TikTok Data

### Imports

In [175]:
import json
import pandas as pd
from pathlib import Path

### Loading JSON files 

In [176]:
folder = Path(
    "/Users/annaclairebreuss-burgess/Library/Mobile Documents/"
    "com~apple~CloudDocs/AC Work/Navigator/"
    "navigator-travel-trends-intelligence/data/tiktok/raw"
)

destinations = [
    "lisbon",
    "marrakech",
    "hanoi",
    "reykjavik"
]

data = {}

for destination in destinations:
    data[destination] = []

    for file_number in [1, 2]:
        file_path = folder / f"{destination}{file_number}.json"

        with open(file_path, "r", encoding="utf-8") as f:
            data[destination].extend(json.load(f))


### Normalising and cleaning data

In [177]:
columns = [
    "url",
    "post_id",
    "description",
    "create_time",
    "digg_count",
    "share_count",
    "collect_count",
    "comment_count",
    "play_count",
    "video_duration",
    "hashtags",
    "profile_username",
    "profile_biography",
    "profile_followers",
    "is_verified",
    "discovery_input.search_keyword",
    "offical_item",
    "original_item",
    "music.authorname",
    "music.original",
    "music.title",
    "region",
    "timestamp"
]

date_columns = [
    "create_time",
    "timestamp"
]

numeric_columns = [
    "digg_count",
    "share_count",
    "collect_count",
    "comment_count",
    "play_count",
    "video_duration",
    "profile_followers"
]

df = {}

for destination in destinations:
    df[destination] = pd.json_normalize(
        data[destination]
    )

    df[destination] = df[destination][columns]

    df[destination]["destination"] = destination

    for column in date_columns:
        df[destination][column] = pd.to_datetime(
            df[destination][column], 
            errors="coerce"
        )

    for column in numeric_columns:
        df[destination][column] = pd.to_numeric(
            df[destination][column], 
            errors="coerce"
        )
    
    print(destination, df[destination].shape)

    df[destination] = df[destination].drop_duplicates(
        subset="post_id"
    )

for destination in destinations:
    print(destination, df[destination].shape)

# print("reykjavik", df[destination].dtypes)
# print("reykjavik", df[destination].head())


lisbon (2098, 24)
marrakech (2103, 24)
hanoi (2123, 24)
reykjavik (2110, 24)
lisbon (875, 24)
marrakech (853, 24)
hanoi (1097, 24)
reykjavik (901, 24)


### Cleaning data - looking at Na values and cleaning hashtags and descriptions

In [178]:
for destination in destinations:
    print(destination)

    print("Na values:")
    print(df[destination].isna().sum())

    print("Number of unique:")
    print("tikToks:", df[destination]["post_id"].nunique())
    print("creators:", df[destination]["profile_username"].nunique())

    df[destination]["hashtags"] = df[destination]["hashtags"].apply(
        lambda x: [hashtag.lower().strip() for hashtag in x]
        if isinstance(x, list)
        else []
    )

    # cleaning description by making everything lowercase and removing #
    df[destination]["description_clean"] = (
    df[destination]["description"]
        .fillna("")
        .str.lower()
        .str.replace(r"#", "", regex=True)
        .str.replace(r"\n", " ", regex=True)
        .str.strip()
    )

    # print(df[destination]["description_clean"].head())



lisbon
Na values:
url                                 0
post_id                             0
description                         3
create_time                         0
digg_count                          0
share_count                        57
collect_count                       0
comment_count                       0
play_count                          0
video_duration                    134
hashtags                           52
profile_username                    0
profile_biography                  38
profile_followers                   0
is_verified                         0
discovery_input.search_keyword      0
offical_item                        0
original_item                       0
music.authorname                    0
music.original                      0
music.title                         0
region                              0
timestamp                           0
destination                         0
dtype: int64
Number of unique:
tikToks: 875
creators: 702
marrakech
Na

url                                 1
post_id                             1
description                         4
create_time                         1
digg_count                          1
share_count                        82
collect_count                       1
comment_count                       1
play_count                          1
video_duration                    116
hashtags                           49
profile_username                    1
profile_biography                  55
profile_followers                   1
is_verified                         1
discovery_input.search_keyword      0
offical_item                        1
original_item                       1
music.authorname                    2
music.original                      1
music.title                         1
region                              1
timestamp                           0
destination                         0
dtype: int64
Number of unique:
tikToks: 900
creators: 708


### Checking and dropping entries with Na post ids

In [179]:
# checking
for destination in destinations:
    print(
        destination,
        df[destination]["post_id"].isna().sum()
    )

# removing
for destination in destinations:
    df[destination] = df[destination].dropna(
        subset=["post_id"]
    )

# checking again
for destination in destinations:
    print(
        destination,
        df[destination]["post_id"].isna().sum()
    )

lisbon 0
marrakech 0
hanoi 0
reykjavik 1
lisbon 0
marrakech 0
hanoi 0
reykjavik 0


### Adding locations mentioned, location types mentioned and classifiers mentioned from tiktoks_locs_clean.json to the destination dataframes

In [180]:
# Load the JSON file
with open(
    "data/tiktok/processed/tiktoks_locs_clean.json",
    "r",
    encoding="utf-8"
) as f:
    locations_data = json.load(f)

# checking the types and values of loc data and df - need to be the same
print(type(locations_data[0]["post_id"]))
# print(locations_data[0]["post_id"])

print(df["lisbon"]["post_id"].dtype)
# print(df["lisbon"]["post_id"].iloc[0])

# adding locations data info ot destinations dfs

# Create lookup dictionary
locations_lookup = {
    post["post_id"]: post["locations"]
    for post in locations_data
}

# Add location information to each destination dataframe
for destination in destinations:

    df[destination]["locations_mentioned"] = (
        df[destination]["post_id"]
        .map(locations_lookup)
        .apply(
            lambda locations: [
                location["name"]
                for location in locations
            ]
            if isinstance(locations, list)
            else []
        )
    )

    df[destination]["location_types_mentioned"] = (
        df[destination]["post_id"]
        .map(locations_lookup)
        .apply(
            lambda locations: [
                location["type"]
                for location in locations
            ]
            if isinstance(locations, list)
            else []
        )
    )

    df[destination]["classifiers_mentioned"] = (
        df[destination]["post_id"]
        .map(locations_lookup)
        .apply(
            lambda locations: list({
                classifier
                for location in locations
                for classifier in location.get("classifiers", [])
            })
            if isinstance(locations, list)
            else []
        )
    )
# check
print(
    df["lisbon"]["locations_mentioned"]
    .loc[
        df["lisbon"]["locations_mentioned"].apply(len) > 0
    ]
    .head()
)
    

<class 'str'>
object
5     [Bessa Restaurante(seafood restaurant )), Mach...
8                                       [Bar Alimentar]
13    [Belém Tower, Jerónimos Monastery, Pastéis de ...
22    [Belém Tower, Jerónimos Monastery, Miradouro d...
23                                             [Chiado]
Name: locations_mentioned, dtype: object


### Turning dfs to csv files (use for analysis)

In [181]:
output_folder = Path(
    "/Users/annaclairebreuss-burgess/Library/Mobile Documents/"
    "com~apple~CloudDocs/AC Work/Navigator/"
    "navigator-travel-trends-intelligence/data/tiktok/processed"
)

output_folder.mkdir(parents=True, exist_ok=True)

for destination, destination_df in df.items():
    destination_df.to_csv(
        output_folder / f"{destination}_clean.csv",
        index=False
    )

### Turning dfs to excel files (use to view)

In [182]:
output_folder = Path(
    "/Users/annaclairebreuss-burgess/Library/Mobile Documents/"
    "com~apple~CloudDocs/AC Work/Navigator/"
    "navigator-travel-trends-intelligence/data/tiktok/processed"
)

output_folder.mkdir(parents=True, exist_ok=True)

output_path = output_folder / "tiktoks_clean.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    for destination, destination_df in df.items():

        # Remove timezone from datetime columns
        # because Excel does not support timezone-aware datetimes
        for column in date_columns:
            destination_df[column] = (
                destination_df[column]
                .dt.tz_localize(None)
            )

        # Write each destination to its own sheet
        destination_df.to_excel(
            writer,
            sheet_name=destination,
            index=False
        )

### Combining dataframes

In [183]:
combined_df = pd.concat(
    df.values(),
    ignore_index=False
)
output_folder = Path(
    "/Users/annaclairebreuss-burgess/Library/Mobile Documents/"
    "com~apple~CloudDocs/AC Work/Navigator/"
    "navigator-travel-trends-intelligence/data/tiktok/processed"
)

output_folder.mkdir(parents=True, exist_ok=True)

combined_df.to_csv(
        output_folder / f"combined_clean.csv",
        index=False
    )

print(combined_df.shape)
combined_df.head()

(3725, 28)


,url,post_id,description,create_time,digg_count,share_count,collect_count,comment_count,play_count,video_duration,...,music.authorname,music.original,music.title,region,timestamp,destination,description_clean,locations_mentioned,location_types_mentioned,classifiers_mentioned
0,https://www.tiktok.com/@alinabadrudin/video/76...,7631556131969666326,I fear Nando’s will never taste the same 😭 #li...,2026-04-22 12:15:33,59400.0,826.0,4213.0,247.0,569000.0,177.0,...,Alina,True,original sound,PT,2026-07-21 10:37:17.446,lisbon,i fear nando’s will never taste the same 😭 lis...,[],[],[]
1,https://www.tiktok.com/@suitelifeoflauren/vide...,7639377582923844894,I planned your entire 48-hour Lisbon itinerary...,2026-05-13 14:06:54,2278.0,330.0,1910.0,27.0,46100.0,68.0,...,Emily,True,original sound,ZA,2026-07-21 10:37:17.511,lisbon,i planned your entire 48-hour lisbon itinerary...,[],[],[]
2,https://www.tiktok.com/@heeral.kakkad/video/76...,7647610856905133333,or like any other city ahhahaha #lisbon #shoul...,2026-06-04 18:36:09,249.0,21.0,5.0,4.0,8966.0,7.0,...,Rod,True,Should I move here,GB,2026-07-21 10:37:17.741,lisbon,or like any other city ahhahaha lisbon shouldi...,[],[],[]
3,https://www.tiktok.com/@cesca.dia/video/763651...,7636514805171490061,Here is a lil recap of what I did in Lisbon to...,2026-05-05 21:30:00,2617.0,267.0,2036.0,62.0,22200.0,145.0,...,Art Bove,True,som original - SuperWave RetroHits,US,2026-07-21 10:37:18.055,lisbon,here is a lil recap of what i did in lisbon to...,[],[],[]
4,https://www.tiktok.com/@babygirls.in.portugal/...,7657857706417556766,Ready for day 2! #shopping #lisbon #girlstrip,2026-07-02 09:19:59,61.0,5.0,2.0,7.0,1321.0,8.0,...,noteliwood,True,original sound,US,2026-07-21 10:37:18.188,lisbon,ready for day 2! shopping lisbon girlstrip,[],[],[]
